# Entrenamiento de la Red Neuronal Profunda
## Proyecto IA #4 - Prediccion de enfermedad cardiaca

Ejecuta las celdas en orden (boton de play o `Shift + Enter`).
Al final descargaras 3 archivos que debes colocar en `backend/model/` de tu repositorio.

### 1. Instalar dependencias

In [ ]:
!pip -q install scikit-learn joblib pandas

### 2. Descargar el dataset

In [ ]:
import urllib.request, pandas as pd
URL = 'https://raw.githubusercontent.com/sharmaroshan/Heart-UCI-Dataset/master/heart.csv'
urllib.request.urlretrieve(URL, 'heart.csv')
df = pd.read_csv('heart.csv')
print('Filas:', df.shape[0], '| Columnas:', df.shape[1])
df.head()

### 3. Preparar los datos

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np, tensorflow as tf
tf.random.set_seed(42); np.random.seed(42)

COLUMNAS = ['age','sex','cp','trestbps','chol','fbs','restecg',
            'thalach','exang','oldpeak','slope','ca','thal']
X = df[COLUMNAS].values
# OJO: en este dataset 'target' esta invertido (target=1 = SANO, target=0 = ENFERMO).
# Invertimos para que nuestra etiqueta 1 signifique 'tiene enfermedad'.
y = 1 - df['target'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
print('Entrenamiento:', X_train.shape, '| Prueba:', X_test.shape)

### 4. Construir la red neuronal profunda (varias capas)

In [ ]:
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping

modelo = models.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(16, activation='relu'),
    layers.Dense(1, activation='sigmoid'),
])
modelo.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
modelo.summary()

### 5. Entrenar

In [ ]:
early = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)
hist = modelo.fit(X_train, y_train, validation_split=0.2,
                  epochs=200, batch_size=16, callbacks=[early], verbose=2)

### 6. Evaluar

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
y_prob = modelo.predict(X_test).ravel()
y_pred = (y_prob >= 0.5).astype(int)
print('Accuracy:', round(accuracy_score(y_test, y_pred), 4))
print('Matriz de confusion:\n', confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

### 7. Guardar y descargar los 3 archivos
Coloca los archivos descargados en `backend/model/` de tu repositorio.

In [ ]:
import json, joblib
modelo.save('modelo_corazon.keras')
joblib.dump(scaler, 'scaler.pkl')
metricas = {
    'accuracy': round(float(accuracy_score(y_test, y_pred)), 4),
    'n_filas': int(df.shape[0]),
    'n_variables': len(COLUMNAS),
    'columnas': COLUMNAS,
    'matriz_confusion': confusion_matrix(y_test, y_pred).tolist(),
    'arquitectura': 'MLP profunda: 64 -> 32 -> 16 -> 1 (sigmoid)'
}
with open('metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metricas, f, indent=2, ensure_ascii=False)

from google.colab import files
files.download('modelo_corazon.keras')
files.download('scaler.pkl')
files.download('metrics.json')